<a href="https://colab.research.google.com/github/JCARNEIROX/IA367-Aprendizado-Reforco/blob/main/Lista2_Ex9_RA239738_RA256389.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IA368FF - Aprendizado por Reforço**  
1º Semestre de 2024  
Prof. Denis Fantinato  

In [ ]:
from copy import deepcopy
import numpy as np

# Criando o MDP

**Grid World 4x3**  
Vamos criar um Grid World 4x3 como um MDP.  

O MDP é definido por:  
                        MDP = (𝑆, 𝐴, 𝑅, ℙ, 𝛾)
com
- 𝑆: conjunto de possíveis estados.
- 𝐴: conjunto de ações.
- 𝑅 ∶ 𝑆 → ℝ: mapa de recompensa para cada estado.
- ℙ: probabilidade de transição de um estado para outro dada uma ação.
- 𝛾: fator de desconto. Um número entre 0 e 1.  
  
Neste mesmo bloco, definimos as probabilidades de transição de estados. Note que o agente tem 80\% de chance de seguir na direção da ação escolhida e 10\% de chance para cada direção perpendicular.

In [ ]:
def createMDP():

    '''
    definição do ambiente
    '''

    S       = [(i,j) for i in range(1,5)
                     for j in range(1,4) if (i,j) != (2,2)]

    goals   = [(4,3), (4,2)]
    actions = ["UP", "DOWN", "LEFT", "RIGHT"]
    A       = {s : actions
               for s in S }

    R        = {s : -0.04 for s in S}
    R[(4,3)] =  1
    R[(4,2)] = -1

    P        = { (s,a) : pvals(s, a, S) for s in S for a in A[s] }

    gamma    = .9

    return (S,A,R,P,gamma)


def move(s, a, S):
    i, j = s
    if a == "UP":
        sp = (i, j+1)
    elif a == "DOWN":
        sp = (i, j-1)
    elif a == "LEFT":
        sp = (i-1, j)
    elif a == "RIGHT":
        sp = (i+1, j)
    elif a is None:
        return s

    if sp in S:
        return sp

    return s

def succ(a):
    return {"UP": "RIGHT", "DOWN": "LEFT", "RIGHT": "DOWN", "LEFT": "UP", None : None}[a]

def pred(a):
    return {"UP": "LEFT", "DOWN": "RIGHT", "RIGHT": "UP", "LEFT": "DOWN", None : None}[a]

def pvals(s, a, S):
    return [(0.8, move(s, a, S)), (0.1, move(s, succ(a), S)), (0.1, move(s, pred(a), S))]


Para confirmar se entendeu esse bloco:    
- Onde a recompensa pode ser ajustada?

  `R:` A recompensa pode ser ajustada trocando o valor "-0.04" na linha de código 15.
- Como ficaria a lista `S`? Escreva os estados na ordem correta.

  `R:` S = [(1, 1),(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2), (3, 3), (4, 1), (4, 2), (4, 3)]

- O que há em:
  - `A[(3,2)]`?

    `R:` A[(3,2)] = ['UP', 'DOWN', 'LEFT', 'RIGHT']

  - `A[(2,2)]`?

    `R:` A[(2,2)] = State (2,2) does not exist

  - `A[(4,2)]`?

    `R:` A[(4,2)] = ['UP', 'DOWN', 'LEFT', 'RIGHT']
- Qual a saída para `P[((3,3),"RIGHT")] = pvals((3,3), "RIGHT", S)`?

    `R:`P[((3,3),"RIGHT")] = [(0.8, (4, 3)), (0.1, (3, 2)), (0.1, (3, 3))]    
- O que acontece se o resultado for um estado fora do grid?

    `R:` O Agente não se movimenta e permanece no estado atual.


# Algoritmo Iteração-de-Valor

Processo iterativo para determinar os valores de utilidade de cada estado. Seja 𝑈𝑖(𝑠) a utilidade para o estado 𝑠 na 𝑖-ésima iteração. Cada iteração atualiza os valores de 𝑈 como:  

𝑈𝑖+1(𝑠) = 𝑅(𝑠) + 𝛾 max𝑎∈𝐴(𝑠)∑𝑠′𝑃(𝑠′∣ 𝑠, 𝑎)𝑈𝑖(𝑠′)  

em que a atualização é feita simultaneamente para todos os estados.

In [ ]:
def valueIteration(mdp, eps):

    S, A, R, P, gamma = mdp

    # Utilidade inicia começa com tudo 0
    Uprime = { s : 0.0 for s in S }

    # delta inicial é o limite + 1 para entrar no laço
    delta  = eps*(1 - gamma)/gamma + 1.

    while delta > eps*(1 - gamma)/gamma:
        U     = deepcopy(Uprime)
        delta = 0
        for s in S:
            if None in A[s]:
                Uprime[s] = R[s]
            else:
                Uprime[s] = R[s] + gamma * max(expVal(P[(s,a)],U) for a in A[s])
            delta     = max(delta, abs(Uprime[s] - U[s]))

    return U

def expVal(ps, U):
    '''
    retorna o valor esperado da utilidade dadas as probabilidades
    de possíveis estados consequentes armazenados em ps.
    '''
    return sum(p*U[s] for p, s in ps)

Para confirmar se entendeu esse bloco:
- Qual a saída de `expVal(P[((3,3),"RIGHT")],U)`, assumindo `U[(3,2)]=1/2`, `U[(3,4)]=1/4` e `U[(3,3)]=1`?

# Algoritmo Iteração-de-Política

Uma outra abordagem é estimar a melhor política e determinar o valor de U a partir dela através de dois passos:  

- Avaliação de política: dada uma política 𝜋, calcule 𝑈, assumindo que 𝜋 é
ótimo.
- Melhoria da política: atualize a política 𝜋 baseado nos valores de 𝑈.  


In [ ]:
def policyIteration(mdp):

    S, A, R, P, gamma = mdp

    U       = { s : 0.0 for s in S }
    # política inicial executa a primeira ação da lista
    pi      = { s : A[s][0] for s in S }
    changed = True

    while changed:
        U       = policyEvaluation(pi, U, mdp)
        changed = False
        for s in S:
            # maior valor esperado utilizando a matriz utilidade atual
            maxU = max(expVal(P[(s,a)],U) for a in A[s])
            # maior valor esperado utilizando a política atual
            piU  = expVal(P[(s,pi[s])],U)

            # Se a utilidade ganhar, atualiza a política
            if maxU > piU:
                idx     = np.argmax( [expVal(P[(s,a)],U) for a in A[s]] )
                pi[s]   = A[s][idx]
                changed = True
                #print(s, maxU, piU)
    return pi

def policyEvaluation(pi, U, mdp):
    S, A, R, P, gamma = mdp
    for s in S:
        U[s] = R[s] + gamma * expVal(P[(s, pi[s])],U)
    return U


Para confirmar se entendeu esse bloco:  
- O que há em `pi[(3,3)]]`?
- Faça a iteração inicial para `s = (3,3)` considerando:
  - `expVal(P[((3,3), "UP")],U) = 0.064`
  - `expVal(P[((3,3), "DOWN")],U) = 0.064`
  - `expVal(P[((3,3), "LEFT")],U) = -0.04`
  - `expVal(P[((3,3), "RIGHT")],U) = 0.792`

# Comparação dos Algoritmos

Função principal:

In [ ]:
def main():
    mdp = createMDP()
    U  = valueIteration(mdp, 1e-3)
    pi = policyIteration(mdp)

    print("Value Iteration: \n")
    for j in range(3,0,-1):
        for i in range(1,5):
            if (i,j) in U:
                print(f"|  {U[(i,j)]:.2f}  |", end="")
            else:
                print("|       |", end="")
        print("")

    print("\n\nPolicy Iteration: \n")
    for j in range(3,0,-1):
        for i in range(1,5):
            if (i,j) in U:
                print(f"|  {pi[(i,j)]}  |", end="")
            else:
                print("|   |", end="")
        print("")

main()